In [1]:

import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [4]:
# Step 1: Create text corpus
corpus = """
The old clock on the mantle ticked slowly, each swing a steady reminder of time passing.
Dust motes danced in the sunlight streaming through the windowpanes, illuminating the quiet room.
A cat, curled on a velvet cushion, twitched its tail in a dream.
Outside, the wind whispered through the leaves of ancient trees,
carrying the scent of damp earth and distant rain.
A book lay open on the table, its pages filled with stories of faraway lands and brave heroes.
The air was still, thick with the unspoken memories held within the walls.
It was a moment of peace, a pause in the hurried rhythm of the world,
where time seemed to stretch and soften, allowing the soul to catch its breath.
The world outside continued its endless turning, but here, in this small corner,
a quiet magic held sway.
"""

In [5]:
# Step 2: Preprocessing
corpus = corpus.lower().replace("\n", " ")

tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts([corpus])

total_chars = len(tokenizer.word_index) + 1


In [6]:
# Step 3: Convert text into sequences
encoded = tokenizer.texts_to_sequences([corpus])[0]

In [7]:
# Step 4: Prepare input-output sequences
seq_length = 40
sequences = []

for i in range(seq_length, len(encoded)):
    seq = encoded[i-seq_length:i]
    label = encoded[i]
    sequences.append((seq, label))

X = np.array([s[0] for s in sequences])

y = to_categorical(
    [s[1] for s in sequences],
    num_classes=total_chars
)


In [8]:
# Step 5: Build LSTM model
model = Sequential([
    Embedding(total_chars, 50, input_length=seq_length),
    LSTM(128, return_sequences=False),
    Dense(total_chars, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [13]:
# Step 6: Compile model
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [14]:
# Step 7: Train model
model.fit(X, y, epochs=40, verbose=1)


Epoch 1/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.7881 - loss: 0.8495
Epoch 2/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.8132 - loss: 0.7736
Epoch 3/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.8517 - loss: 0.7066
Epoch 4/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.8583 - loss: 0.6651
Epoch 5/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.8768 - loss: 0.6305
Epoch 6/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.8993 - loss: 0.5960
Epoch 7/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step - accuracy: 0.9166 - loss: 0.5551
Epoch 8/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - accuracy: 0.9285 - loss: 0.5125
Epoch 9/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9444 - loss: 0.4765
Epoch 10/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9603 - loss: 0.4373
Epoch 11/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9669 - loss: 0.4056
Epoch 12/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy:

In [15]:
# Step 8: Function to generate text
def generate_text(model, tokenizer, seq_length, seed_text, n_chars):

    result = []
    in_text = seed_text.lower()

    for _ in range(n_chars):

        encoded = tokenizer.texts_to_sequences([in_text])[0]
        encoded = encoded[-seq_length:]

        encoded = tf.keras.preprocessing.sequence.pad_sequences(
            [encoded],
            maxlen=seq_length,
            padding='pre'
        )

        y_pred = np.argmax(
            model.predict(encoded, verbose=0),
            axis=1
        )[0]

        out_char = tokenizer.index_word.get(y_pred, '')

        in_text += out_char
        result.append(out_char)

    return seed_text + ''.join(result)

In [16]:
# Step 9: Generate text
print(generate_text(
    model,
    tokenizer,
    seq_length,
    "The old clock",
    100
))


The old clock, with butes dalls ont entin ded stint ranther of tail will stsrering rher heres sfyald, whing the s
